In [4]:
import os
import numpy as np
import pandas as pd
import joblib

# Random Forest Models
from sklearn.ensemble import RandomForestRegressor
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')

In [6]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [7]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet')).iloc[:, 0]
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet')).iloc[:, 0]
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet')).iloc[:, 0]

In [8]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [9]:
print(ID_test.head())
print(ID_test.columns)

  kabupaten_kota_asli  tahun
0       Kab. Balangan   2024
1       Kab. Balangan   2025
2     Kota Balikpapan   2024
3     Kota Balikpapan   2025
4         Kab. Banjar   2024
Index(['kabupaten_kota_asli', 'tahun'], dtype='object')


In [10]:
# Load metadata kolom (referensi)   
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))

In [11]:
# Verifikasi shape
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, ID_train: {ID_train.shape}")
print(f"X_val : {X_val.shape}, y_val : {y_val.shape}, ID_val : {ID_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}, ID_test : {ID_test.shape}")
print(f"\nTotal baris: {len(X_train) + len(X_val) + len(X_test)}")
print(f"Total fitur: {X_train.shape[1]} kolom")
print(f"Target : produktivitas padi (Qu/Ha)")
print(f"Range target train: {y_train.min():.2f} - {y_train.max():.2f}")

X_train: (278, 111), y_train: (278,), ID_train: (278, 2)
X_val : (56, 111), y_val : (56,), ID_val : (56, 2)
X_test : (111, 111), y_test : (111,), ID_test : (111, 2)

Total baris: 445
Total fitur: 111 kolom
Target : produktivitas padi (Qu/Ha)
Range target train: 15.78 - 54.46


In [12]:
# Setup Output Folder
MODEL_FOLDER = os.path.join(BASE_PATH, 'models')
RESULT_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

In [13]:
print("Mulai training Random Forest")

# random forest
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

# training
rf_model.fit(X_train, y_train)

print("Training selesai")

Mulai training Random Forest
Training selesai


In [14]:
# Prediksi
rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

In [15]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [16]:
# evaluasi train
rf_train_mae, rf_train_rmse, rf_train_r2 = hitung_metrics(
    y_train,
    rf_train_pred
)

In [17]:
# evaluasi validation
rf_val_mae, rf_val_rmse, rf_val_r2 = hitung_metrics(
    y_val,
    rf_val_pred
)

In [18]:
# evaluasi test
rf_test_mae, rf_test_rmse, rf_test_r2 = hitung_metrics(
    y_test,
    rf_test_pred
)

In [19]:
print("HASIL RIDGE REGRESSION")

print("\nTrain")
print(f"MAE  : {rf_train_mae:.4f}")
print(f"RMSE : {rf_train_rmse:.4f}")
print(f"R2   : {rf_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {rf_val_mae:.4f}")
print(f"RMSE : {rf_val_rmse:.4f}")
print(f"R2   : {rf_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {rf_test_mae:.4f}")
print(f"RMSE : {rf_test_rmse:.4f}")
print(f"R2   : {rf_test_r2:.4f}")

HASIL RIDGE REGRESSION

Train
MAE  : 1.3301
RMSE : 1.6934
R2   : 0.9380

Validation
MAE  : 3.4727
RMSE : 4.4112
R2   : 0.5474

Test
MAE  : 3.8270
RMSE : 4.8193
R2   : 0.5666


In [20]:
# Simpan Model Ridge Regression
joblib.dump(
    rf_model,
    os.path.join(BASE_PATH, 'models', 'random_forest.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [21]:
# Simpan metrics
metrics_rf = pd.DataFrame({
    "model": ["Random Forest"],
    "train_mae": [rf_train_mae],
    "train_rmse": [rf_train_rmse],
    "train_r2": [rf_train_r2],
    "val_mae": [rf_val_mae],
    "val_rmse": [rf_val_rmse],
    "val_r2": [rf_val_r2],
    "test_mae": [rf_test_mae],
    "test_rmse": [rf_test_rmse],
    "test_r2": [rf_test_r2]
})


os.makedirs("results", exist_ok=True)

metrics_rf.to_csv(
    os.path.join(BASE_PATH, "results", "random_forest_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [22]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_rf"] = rf_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "random_forest_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
  kabupaten_kota_asli  tahun  actual  prediction_rf
0       Kab. Balangan   2024   36.24      37.428233
1       Kab. Balangan   2025   38.35      42.035467
2     Kota Balikpapan   2024   25.34      36.750517
3     Kota Balikpapan   2025   39.36      33.509185
4         Kab. Banjar   2024   37.79      38.598300


In [23]:
feature_importance_rf = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

feature_importance_rf = feature_importance_rf.sort_values(
    by="importance",
    ascending=False
)

feature_importance_rf.to_csv(
    os.path.join(BASE_PATH, "results", 'random_forest_feature_importance.csv'),
    index=False
)

print("Hasil RF feature importance berhasil disimpan.")

Hasil RF feature importance berhasil disimpan.
